## Task 1.1(a): instalar las librerías en el entorno del kernel.

In [16]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

carpeta_datos = Path("datos")

# Cargar los datasets completos
hospitales = gpd.read_file(carpeta_datos / "hospitales_eeuu.geojson")
condados = gpd.read_file(carpeta_datos / "condados_eeuu.geojson")
poblacion = pd.read_csv(carpeta_datos / "poblacion_condados.csv")
estados = gpd.read_file(carpeta_datos / "estados_eeuu.geojson")

archivos = {
    "hospitales_eeuu.geojson": hospitales,
    "condados_eeuu.geojson": condados,
    "poblacion_condados.csv": poblacion,
    "estados_eeuu.geojson": estados,
}

for nombre, datos in archivos.items():
    print(f"Archivo: {nombre}")
    print(f"Número de registros: {len(datos):,}")

    if isinstance(datos, gpd.GeoDataFrame):
        print(f"CRS: {datos.crs}")
    else:
        print("CRS: No aplica (tabla sin geometría)")

    print("Columnas:", ", ".join(datos.columns))
    print()

Archivo: hospitales_eeuu.geojson
Número de registros: 7,154
CRS: EPSG:4326
Columnas: nombre, tipo, ciudad, estado, condado, fips_condado, camas_total, camas_icu, camas_licenciadas, ocupacion_total, ocupacion_icu, geometry

Archivo: condados_eeuu.geojson
Número de registros: 3,221
CRS: EPSG:4326
Columnas: fips, nombre, cod_estado, geometry

Archivo: poblacion_condados.csv
Número de registros: 3,246
CRS: No aplica (tabla sin geometría)
Columnas: fips, condado, estado, lat, lon, poblacion

Archivo: estados_eeuu.geojson
Número de registros: 51
CRS: EPSG:4326
Columnas: nombre, codigo, pais, geometry



## Task 1.1(b): aplicar las limpiezas antes de filtrar por estado.

In [17]:
#Limpiar la tabla de población y normalizar el código 
fips_numerico = pd.to_numeric(poblacion["fips"], errors="raise")

poblacion_limpia = poblacion.loc[fips_numerico < 80000].copy()
poblacion_limpia["fips"] = (
    fips_numerico.loc[poblacion_limpia.index]
    .astype("int64")
    .astype(str)
    .str.zfill(5)
)

print(f"Registros de población originales: {len(poblacion):,}")
print(f"Registros eliminados: {len(poblacion) - len(poblacion_limpia):,}")
print(f"Registros de población restantes: {len(poblacion_limpia):,}")

#Reportar camas_total nulas 
sin_camas = hospitales["camas_total"].isna()
porcentaje_nulos = sin_camas.mean() * 100

print(f"\nHospitales originales: {len(hospitales):,}")
print(f"Hospitales sin datos de camas totales: {sin_camas.sum():,}")
print(f"Porcentaje con camas_total nulo: {porcentaje_nulos:.2f}%")

#Conservar hospitales con datos disponibles.
hospitales_limpios = hospitales.loc[~sin_camas].copy()

print(f"Hospitales después de la limpieza: {len(hospitales_limpios):,}")

#Contar hospitales disponibles 
hospitales_por_estado = (
    hospitales_limpios.groupby("estado")
    .size()
    .reset_index(name="Hospitales con datos de camas")
    .sort_values("Hospitales con datos de camas", ascending=False)
    .reset_index(drop=True)
)

estados_candidatos = hospitales_por_estado.loc[
    hospitales_por_estado["Hospitales con datos de camas"] >= 50
]

print("\nEstados con al menos 50 hospitales después de la limpieza:")
display(estados_candidatos)

Registros de población originales: 3,246
Registros eliminados: 102
Registros de población restantes: 3,144

Hospitales originales: 7,154
Hospitales sin datos de camas totales: 1,090
Porcentaje con camas_total nulo: 15.24%
Hospitales después de la limpieza: 6,064

Estados con al menos 50 hospitales después de la limpieza:


,estado,Hospitales con datos de camas
0,TX,591
1,CA,440
2,FL,305
3,OH,235
4,PA,231
5,NY,206
6,IL,194
7,LA,190
8,GA,164
9,IN,158
